# PINN Thermal Surrogate — Geometry 2 (2a / 2b / 2c)

**What this trains:** A Physics-Informed Neural Network (FourierPINN) for a 3D stacked-die geometry with TSV arrays. Three variants share the same layer structure (10 layers, 8mm × 8mm die) but differ in TSV density:

| Variant | TSV density | Effective k_TSV |
|---------|------------|------------------|
| geometry2a | 3% | ~7.3 W/m·K |
| geometry2b | 5% | ~11.1 W/m·K |
| geometry2c | 10% | ~20.8 W/m·K |

**Key difference from geometry1:** `fourier_sigma=20.0` (captures TSV-scale thermal features at ~500 µm; geometry1 uses 10.0).

**Architecture:** Same FourierPINN, but `n_layers=10` (9 layers + TIM2 topmost).

## Kaggle Dataset Setup

Add **two datasets** to this notebook:

1. **Source code** — upload the `src/` directory as a Kaggle dataset (slug: `thermo-pinn-src`)
2. **3D-ICE training data** — upload the `.npz` files from `data/3d-ice/` (slug: `3dice-thermal-data`)

To train all three variants, **run this notebook three times** — change only `GEOM_NAME` in the Config cell.  
Alternatively, set `TRAIN_ALL_VARIANTS = True` to train 2a → 2b → 2c sequentially in one session.

**Expected training time per variant on T4 GPU:** ~3–5 hours (8000 epochs, 15 training scenarios, mesh 80×80×72)

In [ ]:
import subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml'], check=True)

import os
import torch

print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
import sys
from pathlib import Path

# ── Kaggle dataset paths ─────────────────────────────────────────────────────
SRC_DATASET  = 'thermo-pinn-src'
DATA_DATASET = '3dice-thermal-data'

SRC_ROOT  = Path(f'/kaggle/input/{SRC_DATASET}')
DATA_ROOT = Path(f'/kaggle/input/{DATA_DATASET}')

sys.path.insert(0, str(SRC_ROOT))

assert SRC_ROOT.exists(),  f'Source dataset not found: {SRC_ROOT}'
assert DATA_ROOT.exists(), f'Data dataset not found: {DATA_ROOT}'
print('Source root:', SRC_ROOT)
print('Data root:  ', DATA_ROOT)

In [ ]:
import logging
import numpy as np
import matplotlib.pyplot as plt

from src.core.geometry_builders import get_geometry_by_name
from src.pinn.data_loader import ThermalDataset, NormStats, compute_norm_stats
from src.pinn.model import build_model
from src.pinn.trainer import Trainer

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)-8s %(name)s: %(message)s',
    datefmt='%H:%M:%S',
)
print('Imports OK')

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
# Change GEOM_NAME to 'geometry2a', 'geometry2b', or 'geometry2c'
GEOM_NAME      = 'geometry2a'
TRAIN_ALL_VARIANTS = False  # Set True to train 2a→2b→2c sequentially in one run

FOURIER_SIGMA  = 20.0    # Higher than geometry1: captures TSV-scale features (~500 µm)
HIDDEN_DIM     = 256
N_RES_BLOCKS   = 6
N_COL          = 20000
EPOCHS         = 8000
LR             = 3e-4
LOG_INTERVAL   = 200
VAL_INTERVAL   = 1       # validate + save best checkpoint every N epochs
                         # 1 = every epoch — checkpoint stays current if session crashes

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training on: {DEVICE}')

VARIANTS = ['geometry2a', 'geometry2b', 'geometry2c'] if TRAIN_ALL_VARIANTS else [GEOM_NAME]
print(f'Will train: {VARIANTS}')

In [ ]:
# ── Helper functions ──────────────────────────────────────────────────────────
def collect_files(data_dir, geom, split):
    files = sorted(data_dir.rglob(f'{geom}_{split}_*.npz'))
    if not files:
        files = sorted(data_dir.glob(f'{geom}_{split}_*.npz'))
    return files


def train_one_variant(geom_name: str, results_log: list) -> dict:
    print(f'\n{"="*60}')
    print(f'Training PINN for {geom_name}')
    print(f'{"="*60}')

    geometry  = get_geometry_by_name(geom_name)
    geometries = {geom_name: geometry}
    out_dir   = Path(f'/kaggle/working/checkpoints/{geom_name}')
    out_dir.mkdir(parents=True, exist_ok=True)

    train_files = collect_files(DATA_ROOT, geom_name, 'train')
    test_files  = collect_files(DATA_ROOT, geom_name, 'test')
    assert train_files, f'No training files for {geom_name}'
    print(f'Files: {len(train_files)} train + {len(test_files)} test')

    # Norm stats
    norm_path = out_dir / 'norm_stats.json'
    if norm_path.exists():
        norm_stats = NormStats.load(norm_path)
        print('Loaded norm stats')
    else:
        norm_stats = compute_norm_stats(train_files, geometries)
        norm_stats.save(norm_path)
        print('Computed norm stats')

    train_dataset = ThermalDataset(train_files, geometries, norm_stats)
    val_files     = test_files if test_files else train_files[-3:]
    val_dataset   = ThermalDataset(val_files, geometries, norm_stats)

    n_layers = len(geometry.layers)
    print(f'Layers: {n_layers} | TSV density: {geometry.tsv_density*100:.0f}%')
    print(geometry.summary())

    model = build_model(
        n_layers=n_layers,
        fourier_sigma=FOURIER_SIGMA,
        hidden_dim=HIDDEN_DIM,
        n_res_blocks=N_RES_BLOCKS,
        device=DEVICE,
    )
    n_params = sum(p.numel() for p in model.parameters())
    print(f'FourierPINN: {n_params:,} parameters')

    trainer = Trainer(
        model=model,
        geometry=geometry,
        norm_stats=norm_stats,
        train_data=train_dataset,
        val_data=val_dataset,
        output_dir=out_dir,
        n_col=N_COL,
        epochs=EPOCHS,
        lr=LR,
        device=DEVICE,
        log_interval=LOG_INTERVAL,
        val_interval=VAL_INTERVAL,
    )

    best_ckpt = trainer.train()

    result = {
        'geom_name':     geom_name,
        'tsv_density':   geometry.tsv_density,
        'n_params':      n_params,
        'best_val_mae_K': trainer.best_val_mae,
        'checkpoint':    str(best_ckpt),
        'history':       trainer.history,
    }
    results_log.append(result)
    return result


print('Helper functions defined')

In [ ]:
# ── Train ─────────────────────────────────────────────────────────────────────
all_results = []
for variant in VARIANTS:
    res = train_one_variant(variant, all_results)
    print(f'\n{variant} best val MAE: {res["best_val_mae_K"]:.3f} K')

print('\nAll training runs complete.')

In [ ]:
# ── Training curves (one panel per trained variant) ───────────────────────────
n_variants = len(all_results)
fig, axes = plt.subplots(n_variants, 3, figsize=(15, 4 * n_variants), squeeze=False)
fig.suptitle('PINN Training Curves — Geometry 2', fontsize=13)

for row, res in enumerate(all_results):
    h = res['history']
    ep = h['epoch']
    gname = res['geom_name']

    axes[row, 0].semilogy(ep, h['train_data'], label='Data', color='steelblue')
    axes[row, 0].semilogy(ep, h['train_pde'],  label='PDE',  color='tomato')
    axes[row, 0].semilogy(ep, h['train_bc'],   label='BC',   color='forestgreen')
    axes[row, 0].set_title(f'{gname} — Loss components')
    axes[row, 0].set_xlabel('Epoch'); axes[row, 0].set_ylabel('Loss')
    axes[row, 0].legend(); axes[row, 0].grid(alpha=0.3)

    axes[row, 1].semilogy(ep, h['train_total'], color='darkorange')
    axes[row, 1].set_title(f'{gname} — Total loss')
    axes[row, 1].set_xlabel('Epoch'); axes[row, 1].grid(alpha=0.3)

    val_maes = h['val_mae_K']
    best_mae = min(val_maes)
    best_ep  = ep[val_maes.index(best_mae)]
    axes[row, 2].plot(ep, val_maes, color='purple')
    axes[row, 2].axhline(best_mae, linestyle='--', color='purple', alpha=0.5,
                         label=f'Best {best_mae:.2f} K @ {best_ep}')
    axes[row, 2].set_title(f'{gname} — Val MAE')
    axes[row, 2].set_xlabel('Epoch'); axes[row, 2].set_ylabel('MAE (K)')
    axes[row, 2].legend(); axes[row, 2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/checkpoints/geometry2_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Evaluation ────────────────────────────────────────────────────────────────
import json

def evaluate_variant(res: dict) -> dict:
    geom_name  = res['geom_name']
    checkpoint = res['checkpoint']
    geometry   = get_geometry_by_name(geom_name)
    geometries = {geom_name: geometry}
    out_dir    = Path(f'/kaggle/working/checkpoints/{geom_name}')

    norm_stats = NormStats.load(out_dir / 'norm_stats.json')
    T_range    = norm_stats.T_max - norm_stats.T_min

    test_files = collect_files(DATA_ROOT, geom_name, 'test')
    val_files  = test_files if test_files else collect_files(DATA_ROOT, geom_name, 'train')[-3:]
    val_dataset = ThermalDataset(val_files, geometries, norm_stats)
    val_dataset.to_device(DEVICE)

    n_layers = len(geometry.layers)
    model = build_model(
        n_layers=n_layers,
        fourier_sigma=FOURIER_SIGMA,
        hidden_dim=HIDDEN_DIM,
        n_res_blocks=N_RES_BLOCKS,
        device=DEVICE,
    )
    ckpt = torch.load(checkpoint, map_location=DEVICE)
    model.load_state_dict(ckpt['model_state'])
    model.eval()

    maes = []
    print(f'\n{geom_name} test results (TSV={geometry.tsv_density*100:.0f}%):')
    print(f'{"Scenario":<35} {"MAE(K)":>8} {"P95(K)":>8} {"Max(K)":>8}')
    print('-' * 65)
    with torch.no_grad():
        for sc in val_dataset.scenarios:
            htc_t  = torch.tensor(sc.htc_norm,   dtype=torch.float32, device=DEVICE)
            tamb_t = torch.tensor(sc.t_amb_norm, dtype=torch.float32, device=DEVICE)
            tsv_t  = torch.tensor(sc.tsv_frac,   dtype=torch.float32, device=DEVICE)
            T_pred = model(sc.coords, sc.layer_ids, sc.power, htc_t, tamb_t, tsv_t)
            T_pred_K = T_pred.cpu().numpy() * T_range + norm_stats.T_min
            T_true_K = sc.temp.cpu().numpy() * T_range + norm_stats.T_min
            err = np.abs(T_pred_K - T_true_K)
            mae = err.mean()
            maes.append(mae)
            print(f'{sc.name:<35} {mae:>8.2f} {np.percentile(err,95):>8.2f} {err.max():>8.2f}')

    print('-' * 65)
    print(f'{"MEAN":<35} {np.mean(maes):>8.2f}')
    return {'geom_name': geom_name, 'tsv_density': geometry.tsv_density,
            'mean_mae_K': float(np.mean(maes)), 'max_mae_K': float(np.max(maes))}


eval_results = [evaluate_variant(r) for r in all_results]

In [ ]:
# ── TSV density comparison plot ───────────────────────────────────────────────
if len(eval_results) > 1:
    tsv_densities = [r['tsv_density'] * 100 for r in eval_results]
    mean_maes     = [r['mean_mae_K'] for r in eval_results]
    max_maes      = [r['max_mae_K'] for r in eval_results]

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(tsv_densities, mean_maes, 'o-', color='steelblue', label='Mean MAE')
    ax.plot(tsv_densities, max_maes,  's--', color='tomato',   label='Max MAE')
    ax.set_xlabel('TSV density (%)')
    ax.set_ylabel('MAE (K)')
    ax.set_title('PINN accuracy vs TSV density')
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('/kaggle/working/checkpoints/geometry2_tsv_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

# Summary table
print('\nSummary across all trained variants:')
print(f'{"Geometry":<15} {"TSV%":>6} {"Mean MAE (K)":>14} {"Max MAE (K)":>12}')
print('-' * 52)
for r in eval_results:
    print(f'{r["geom_name"]:<15} {r["tsv_density"]*100:>6.0f} {r["mean_mae_K"]:>14.2f} {r["max_mae_K"]:>12.2f}')

In [ ]:
# ── Save summary ──────────────────────────────────────────────────────────────
import json

summary = {
    'fourier_sigma':  FOURIER_SIGMA,
    'hidden_dim':     HIDDEN_DIM,
    'n_res_blocks':   N_RES_BLOCKS,
    'epochs':         EPOCHS,
    'variants': [
        {
            'geom_name':      r['geom_name'],
            'tsv_density':    r['tsv_density'],
            'best_val_mae_K': ar['best_val_mae_K'],
            'test_mean_mae_K': er['mean_mae_K'],
            'test_max_mae_K':  er['max_mae_K'],
        }
        for ar, er in zip(all_results, eval_results)
        for r in [er]
    ]
}

with open('/kaggle/working/checkpoints/geometry2_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))